# Phase 4 — Collaborative Filtering

## Objective

Build a personalized movie recommendation model using user-rating behavior.

### Technique
- Singular Value Decomposition (SVD)
- Surprise library
- Train/Test Split
- RMSE
- MAE

The model learns latent relationships between users and movies and predicts ratings for movies a user has not rated.


In [1]:
import pandas as pd
import numpy as np

from surprise import Dataset, Reader, SVD, accuracy
from surprise.model_selection import train_test_split


In [2]:
ratings = pd.read_csv("../data/ratings.csv")

print("Ratings Shape:", ratings.shape)
ratings.head()


Ratings Shape: (105339, 4)


,userId,movieId,rating,timestamp
0,1,16,4.0,1217897793
1,1,24,1.5,1217895807
2,1,32,4.0,1217896246
3,1,47,4.0,1217896556
4,1,50,4.0,1217896523


In [3]:
print(ratings.info())

print("\nMissing Values:")
print(ratings.isnull().sum())

print("\nRating Statistics:")
print(ratings["rating"].describe())


<class 'pandas.DataFrame'>
RangeIndex: 105339 entries, 0 to 105338
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   userId     105339 non-null  int64  
 1   movieId    105339 non-null  int64  
 2   rating     105339 non-null  float64
 3   timestamp  105339 non-null  int64  
dtypes: float64(1), int64(3)
memory usage: 3.2 MB
None

Missing Values:
userId       0
movieId      0
rating       0
timestamp    0
dtype: int64

Rating Statistics:
count    105339.000000
mean          3.516850
std           1.044872
min           0.500000
25%           3.000000
50%           3.500000
75%           4.000000
max           5.000000
Name: rating, dtype: float64


In [4]:
rating_min = float(ratings["rating"].min())
rating_max = float(ratings["rating"].max())

print("Rating Scale:", rating_min, "to", rating_max)

reader = Reader(
    rating_scale=(rating_min, rating_max)
)

data = Dataset.load_from_df(
    ratings[["userId", "movieId", "rating"]],
    reader
)


Rating Scale: 0.5 to 5.0


In [5]:
trainset, testset = train_test_split(
    data,
    test_size=0.20,
    random_state=42
)

print("Training Ratings:", trainset.n_ratings)
print("Test Ratings:", len(testset))


Training Ratings: 84271
Test Ratings: 21068


In [6]:
model = SVD(
    n_factors=100,
    n_epochs=20,
    lr_all=0.005,
    reg_all=0.02,
    random_state=42
)

model.fit(trainset)

print("SVD model training completed.")


SVD model training completed.


In [7]:
predictions = model.test(testset)

predictions[:5]


[Prediction(uid=33, iid=1207, r_ui=5.0, est=np.float64(4.455707166663636), details={'was_impossible': False}),
 Prediction(uid=420, iid=597, r_ui=5.0, est=np.float64(3.556328478502209), details={'was_impossible': False}),
 Prediction(uid=432, iid=1377, r_ui=5.0, est=np.float64(4.319359836238306), details={'was_impossible': False}),
 Prediction(uid=160, iid=1209, r_ui=3.0, est=np.float64(4.282177464067733), details={'was_impossible': False}),
 Prediction(uid=281, iid=1036, r_ui=4.0, est=np.float64(4.076199007720687), details={'was_impossible': False})]

In [8]:
rmse = accuracy.rmse(predictions, verbose=True)
print("RMSE:", rmse)

mae = accuracy.mae(predictions, verbose=True)
print("MAE:", mae)


RMSE: 0.8671
RMSE: 0.8671218020938558
MAE:  0.6719
MAE: 0.6719414745247575


In [9]:
user_id = int(ratings["userId"].iloc[0])
movie_id = int(ratings["movieId"].iloc[0])

prediction = model.predict(user_id, movie_id)

print("User:", user_id)
print("Movie:", movie_id)
print("Predicted Rating:", prediction.est)


User: 1
Movie: 16
Predicted Rating: 3.7769551140901854


In [10]:
movies = pd.read_csv("../data/movies.csv")
movies.head()


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [11]:
def get_rated_movie_ids(user_id):
    return set(
        ratings.loc[
            ratings["userId"] == user_id,
            "movieId"
        ]
    )


In [12]:
def collaborative_recommend(user_id, n=10):
    rated_movie_ids = get_rated_movie_ids(user_id)

    candidates = movies[
        ~movies["movieId"].isin(rated_movie_ids)
    ].copy()

    candidates["predicted_rating"] = candidates["movieId"].apply(
        lambda movie_id: model.predict(
            int(user_id),
            int(movie_id)
        ).est
    )

    recommendations = candidates.sort_values(
        "predicted_rating",
        ascending=False
    ).head(n)

    return recommendations[
        ["movieId", "title", "genres", "predicted_rating"]
    ].reset_index(drop=True)


In [13]:
selected_user = int(ratings["userId"].iloc[0])

collaborative_recommend(selected_user, 10)


,movieId,title,genres,predicted_rating
0,1212,"Third Man, The (1949)",Film-Noir|Mystery|Thriller,4.395319
1,3000,Princess Mononoke (Mononoke-hime) (1997),Action|Adventure|Animation|Drama|Fantasy,4.300890
2,1204,Lawrence of Arabia (1962),Adventure|Drama|War,4.300286
3,923,Citizen Kane (1941),Drama|Mystery,4.295864
4,1248,Touch of Evil (1958),Crime|Film-Noir|Thriller,4.287021
5,26131,"Battle of Algiers, The (La battaglia di Algeri...",Drama|War,4.250366
6,40629,Pride & Prejudice (2005),Drama|Romance,4.244983
7,1250,"Bridge on the River Kwai, The (1957)",Adventure|Drama|War,4.241299
8,2997,Being John Malkovich (1999),Comedy|Drama|Fantasy,4.225687
9,1280,Raise the Red Lantern (Da hong deng long gao g...,Drama,4.215417


In [14]:
recommendations = collaborative_recommend(selected_user, 10)

rated_ids = get_rated_movie_ids(selected_user)
overlap = set(recommendations["movieId"]) & rated_ids

print("Recommended movies already rated:", len(overlap))
print("Overlap:", overlap)


Recommended movies already rated: 0
Overlap: set()


## Interpretation

Collaborative Filtering uses historical user-rating behavior. SVD learns latent user-movie patterns and predicts ratings for candidate movies.

Movies with the highest predicted ratings are returned as personalized recommendations. Already-rated movies are excluded from the recommendation list.


# Phase 4 Conclusion

The Collaborative Filtering model was successfully implemented using SVD.

The model predicts user preferences and produces personalized recommendations. RMSE and MAE are used to evaluate rating-prediction performance.
